In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   CarRacing-v3 PPO Final Submission — Google Colab Notebook           ║
# ║   Self-contained · MlpPolicy · Gymnasium CarRacing-v3        ║
# ╠══════════════════════════════════════════════════════════════╣
# ║  SAFETY: Cells 0–15 contain NO training.                     ║
# ║  Cell 16 requires: DRY_RUN=False AND TRAIN_APPROVED=True     ║
# ║  Do not call model.learn() outside Cells 16/17.              ║
# ║  Eval/video cells load saved artifacts — not in-memory state.║
# ╚══════════════════════════════════════════════════════════════╝
#
# Submission notes:
#   - swig installed before gymnasium[box2d]
#   - automatic Drive folder creation
#   - progress_bar=True in main training
#   - checkpoint + VecNormalize pairing
#   - evaluations.npz Drive-backed backup
#   - eval_live.csv live logging
#   - COMPARE5 seeded eval + RANDOM10 deterministic eval
#   - first-frame hash to verify seed diversity
#   - fresh env per eval seed for independent evaluation
#   - reload-after-disconnect + resume-training cells
#   - training guards (DRY_RUN / TRAIN_APPROVED)
#   - distilled plots that read from saved files
#   - tile_bonus_total and tile_bonus_mean in episode info
#
# Run order: Cell 1 → restart → Cells 2–15 → Cell 16 (training)
#            → Cells 17+ (resume / eval / video / package)

print("CarRacing-v3 PPO final submission notebook.")
print("Run Cell 1, restart runtime, then Cells 2-15 before training.")


In [ ]:
# Cell 1 — Robust dependency install
# Run FIRST. Restart runtime/kernel before any other cell.

import subprocess, sys

def run(cmd, check=True):
    print(f"\n$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout:
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        if check:
            raise RuntimeError(f"Command failed: {cmd}")
    return r

print("Python:", sys.executable)

# System packages — swig must come before gymnasium[box2d]
run("apt-get update -y -qq", check=False)
run("apt-get install -y -qq swig build-essential python3-dev ffmpeg", check=False)
run(f"{sys.executable} -m pip install -q --no-cache-dir swig")

# SB3 2.5+ supports NumPy 2.x / Python 3.12
run(f"{sys.executable} -m pip install -q --upgrade --no-cache-dir 'stable-baselines3>=2.5.0'")
run(f"{sys.executable} -m pip install -q --upgrade --no-cache-dir 'gymnasium[box2d]>=0.29.1'")
run(f"{sys.executable} -m pip install -q --upgrade --no-cache-dir "
    "opencv-python-headless imageio imageio-ffmpeg matplotlib pandas")

print("\n=== Package versions ===")
for pkg in ["numpy", "stable-baselines3", "gymnasium", "opencv-python-headless"]:
    run(f"{sys.executable} -m pip show {pkg}", check=False)

print("\n✅ Install complete.")
print("RESTART RUNTIME NOW, then run Cells 2-15.")


In [ ]:
# Cell 2 — Mount Google Drive and define final output paths

import os

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _DRIVE_OK = True
except Exception:
    _DRIVE_OK = False
    print("WARNING: Drive not available — saving locally only")

DRIVE_ROOT = "/content/drive/MyDrive/CarRacingV3_PPO_Final_Submission"
LOCAL_ROOT  = "/content/carracingv3_ppo_final"

CKPT_DIR     = f"{LOCAL_ROOT}/models/checkpoints"
BEST_DIR     = f"{LOCAL_ROOT}/models/best"
LOG_DIR      = f"{LOCAL_ROOT}/logs"
EVAL_BCK_DIR = f"{LOCAL_ROOT}/logs/eval_backups"
TABLE_DIR    = f"{LOCAL_ROOT}/tables"
FIG_DIR      = f"{LOCAL_ROOT}/figures"
VIDEO_DIR    = f"{LOCAL_ROOT}/videos"
RELEASE_DIR  = f"{LOCAL_ROOT}/release"

for _d in [CKPT_DIR, BEST_DIR, LOG_DIR, EVAL_BCK_DIR,
           TABLE_DIR, FIG_DIR, VIDEO_DIR, RELEASE_DIR]:
    os.makedirs(_d, exist_ok=True)

if _DRIVE_OK:
    for _sub in ["models/checkpoints", "models/best", "logs", "logs/eval_backups",
                 "tables", "figures", "videos", "release"]:
        os.makedirs(f"{DRIVE_ROOT}/{_sub}", exist_ok=True)


def _backup(local_path):
    """Copy local file to Drive mirror."""
    if not _DRIVE_OK or not os.path.isfile(local_path):
        return
    import shutil as _sh
    rel = os.path.relpath(local_path, LOCAL_ROOT)
    dst = f"{DRIVE_ROOT}/{rel}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    _sh.copy2(local_path, dst)


def _kb(path):
    return os.path.getsize(path) / 1024 if os.path.isfile(path) else 0.0


print(f"Local root : {LOCAL_ROOT}")
print(f"Drive root : {DRIVE_ROOT}")
print(f"Drive OK   : {_DRIVE_OK}")
print("Final output folders created.")
print("✅ Cell 2 — paths ready")


In [ ]:
# Cell 3 — Imports and global config

import sys, os, time, hashlib, warnings, shutil
import numpy as np
import cv2
import gymnasium as gym
import stable_baselines3 as sb3
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import imageio

warnings.filterwarnings("ignore", category=UserWarning)

# ── Version gate ──────────────────────────────────────────────
np_major = int(np.__version__.split(".")[0])
sb3_ver  = tuple(int(x) for x in sb3.__version__.split(".")[:2])
if np_major >= 2:
    assert sb3_ver >= (2, 5), (
        f"NumPy {np.__version__} requires SB3 >= 2.5. Got {sb3.__version__}. "
        "Re-run Cell 1 and restart runtime."
    )

import Box2D  # will raise if gymnasium[box2d] not installed

print(f"Python  {sys.version.split()[0]}")
print(f"NumPy   {np.__version__}  |  SB3 {sb3.__version__}")
print(f"gym     {gym.__version__}  |  torch {torch.__version__}")
print(f"CUDA    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     {torch.cuda.get_device_name(0)}")

# ── Environment ──────────────────────────────────────────────
CAR_RACING_ENV_ID = "CarRacing-v3"

# ── Training ─────────────────────────────────────────────────
SEED            = 42
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
TOTAL_TIMESTEPS = 500_000
N_ENVS          = 4
N_STACK         = 4
CROP_BOTTOM     = 84
ACTION_REPEAT   = 2

# ── PPO ──────────────────────────────────────────────────────
INIT_LR         = 3e-4
FINAL_LR        = 1e-5
INIT_CLIP       = 0.20
FINAL_CLIP      = 0.10
ENT_COEF        = 0.01   # SB3 PPO does not accept callable for ent_coef
N_STEPS         = 1024
BATCH_SIZE      = 256
N_EPOCHS        = 10
GAMMA           = 0.99
GAE_LAMBDA      = 0.95
VF_COEF         = 0.5
MAX_GRAD_NORM   = 0.5
USE_SDE         = False
SDE_SAMPLE_FREQ = -1

# ── Wrapper ──────────────────────────────────────────────────
EMA_ALPHA        = 0.65
MAX_OFFROAD      = 25
MAX_STILL        = 50
STILL_SPEED_THR  = 0.03
OFFROAD_PEN      = 0.50
STEER_PEN        = 0.04
GAS_OFFROAD_PEN  = 0.10
TILE_BONUS       = 0.50
STEERING_GUIDE   = 0.15
SPEED_BONUS      = 0.02
CURVE_BRAKE_BON  = 0.08
FRONT_DANGER_PEN = 0.20
ANTI_SPIN_PEN    = 0.30
GRASS_LOW        = np.array([28,  35,  35],  dtype=np.uint8)
GRASS_HIGH       = np.array([92, 255, 255],  dtype=np.uint8)
RAY_ANGLES_DEG   = np.array(
    [-70, -52.5, -35, -17.5, 0, 17.5, 35, 52.5, 70], dtype=np.float32
)
RAY_ORIGIN       = (48, 70)   # (x, y) in cropped image - calibrated origin

# ── Checkpoint / eval frequency ──────────────────────────────
CKPT_FREQ    = 25_000 // N_ENVS
EVAL_FREQ    = 20_000 // N_ENVS
N_EVAL_EPS   = 5

# ── Final evaluation constants ──────────────────────────────────
COMPARE5_SEEDS       = [42, 123, 777, 999, 2026]
RANDOM10_MASTER_SEED = 42
RANDOM10_COUNT       = 10

# ── Training guards (Cell 16/17 check these) ─────────────────
# ponytail: set both to True only when ready to train
DRY_RUN        = False
TRAIN_APPROVED = True

print(f"\nDEVICE={DEVICE}  TOTAL_TIMESTEPS={TOTAL_TIMESTEPS:,}  N_ENVS={N_ENVS}")
print(f"OBS_DIM=14  STACKED={14*N_STACK}  USE_SDE={USE_SDE}")
print(f"DRY_RUN={DRY_RUN}  TRAIN_APPROVED={TRAIN_APPROVED}")
print("✅ Cell 3 — config ready")


In [ ]:
# Cell 4 — Helper functions and schedules

from typing import Callable


def linear_schedule(init_val: float, final_val: float) -> Callable[[float], float]:
    """Linear decay: progress_remaining 1.0->0.0 maps to init_val->final_val."""
    def _s(p: float) -> float:
        return final_val + p * (init_val - final_val)
    return _s


lr_schedule   = linear_schedule(INIT_LR, FINAL_LR)
clip_schedule = linear_schedule(INIT_CLIP, FINAL_CLIP)


def _first_frame_hash(obs_array: np.ndarray) -> str:
    """MD5 of first-frame observation bytes — used to verify seed diversity."""
    return hashlib.md5(np.asarray(obs_array).tobytes()).hexdigest()[:12]


print(f"LR  : {lr_schedule(1.0):.2e} -> {lr_schedule(0.0):.2e}")
print(f"Clip: {clip_schedule(1.0):.3f} -> {clip_schedule(0.0):.3f}")
print("✅ Cell 4 — schedules and helpers ready")


In [ ]:
# Cell 5 — HSV road mask + dynamic ray casting


def road_mask_hsv(obs_rgb: np.ndarray) -> np.ndarray:
    """HSV grass detect -> invert -> road mask. Shape (CROP_BOTTOM, 96), road=255."""
    hsv   = cv2.cvtColor(obs_rgb[:CROP_BOTTOM, :, :], cv2.COLOR_RGB2HSV)
    grass = cv2.inRange(hsv, GRASS_LOW, GRASS_HIGH)
    return cv2.bitwise_not(grass)


def cast_rays(mask: np.ndarray, angles_deg: np.ndarray,
              max_len: float, origin: tuple = RAY_ORIGIN,
              step: float = 2.0) -> np.ndarray:
    """Cast rays from origin, return normalized distances [0, 1]."""
    h, w = mask.shape
    ox, oy = origin
    feats = []
    for angle in angles_deg:
        rad = np.deg2rad(float(angle))
        dx, dy = np.sin(rad), -np.cos(rad)
        dist = 0.0
        while dist < max_len:
            ix = int(ox + dx * dist)
            iy = int(oy + dy * dist)
            if not (0 <= ix < w and 0 <= iy < h):
                break
            if mask[iy, ix] == 0:
                break
            dist += step
        feats.append(min(dist, max_len) / max_len)
    return np.array(feats, dtype=np.float32)


def compute_rays_dynamic(mask: np.ndarray, prev_speed: float = 0.0) -> np.ndarray:
    """Speed-adaptive ray range: speed 0->1 gives max_len 45->80, squeeze 0->18%."""
    spd     = float(np.clip(prev_speed, 0.0, 1.0))
    max_len = 45.0 + spd * 35.0
    squeeze = 1.0 - spd * 0.18
    return cast_rays(mask, RAY_ANGLES_DEG * squeeze, max_len)


print("✅ Cell 5 — HSV/ray helpers ready")


In [ ]:
# Cell 6 — CarRacingRayWrapper
# Feature wrapper with ray observations. Added tile_bonus_total and tile_bonus_mean to info.


class CarRacingRayWrapper(gym.Wrapper):
    """
    Ray-feature wrapper: 14D obs, 3D action [steer, gas, brake], EMA=65%, ACTION_REPEAT=2.
    Obs: 9 dynamic rays + prev_steer + prev_gas + prev_brake + speed + curvature.
    """

    def __init__(self, env: gym.Env):
        super().__init__(env)
        self.action_space = gym.spaces.Box(
            low=np.array([-1.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([ 1.0, 1.0, 1.0], dtype=np.float32),
        )
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=(14,), dtype=np.float32
        )
        self._smoothed              = np.zeros(3, dtype=np.float32)
        self._prev_smoothed_steer   = 0.0
        self._offroad_cnt           = 0
        self._still_cnt             = 0
        self._step_cnt              = 0
        self._prev_tiles            = 0
        self._prev_speed            = 0.0
        self._raw_ep_rew            = 0.0
        self._tile_bonus_total      = 0.0

    def _get_speed(self) -> float:
        try:
            v = self.unwrapped.car.hull.linearVelocity
            return float(np.sqrt(v[0]**2 + v[1]**2)) / 30.0
        except Exception:
            return 0.0

    def _build_obs(self, obs_rgb, speed):
        mask = road_mask_hsv(obs_rgb)
        rays = compute_rays_dynamic(mask, prev_speed=speed)
        fa   = float(np.mean(rays[3:6]))
        la   = float(np.mean(rays[:4]))
        ra   = float(np.mean(rays[5:]))
        on_road   = fa > 0.20
        curvature = (float(np.clip(abs(la - ra) * (1.0 - rays[4]), 0.0, 1.0))
                     if on_road else 0.0)
        obs_vec = np.array(
            [*rays, self._smoothed[0], self._smoothed[1], self._smoothed[2],
             speed, curvature],
            dtype=np.float32,
        )
        return obs_vec, rays, on_road, curvature, fa, la, ra

    def reset(self, **kwargs):
        self._smoothed            = np.zeros(3, dtype=np.float32)
        self._prev_smoothed_steer = 0.0
        self._offroad_cnt         = 0
        self._still_cnt           = 0
        self._step_cnt            = 0
        self._prev_tiles          = 0
        self._prev_speed          = 0.0
        self._raw_ep_rew          = 0.0
        self._tile_bonus_total    = 0.0
        obs_rgb, info = self.env.reset(**kwargs)
        speed = self._get_speed()
        self._prev_speed = speed
        obs_vec, _, _, _, _, _, _ = self._build_obs(obs_rgb, speed)
        return obs_vec, info

    def step(self, action: np.ndarray):
        # Save prev steer BEFORE EMA update (uses previous steering before smoothing)
        self._prev_smoothed_steer = float(self._smoothed[0])

        clipped = np.clip(action, self.action_space.low, self.action_space.high)
        self._smoothed = (
            (1.0 - EMA_ALPHA) * self._smoothed + EMA_ALPHA * clipped
        ).astype(np.float32)

        full_action = np.array(
            [self._smoothed[0], self._smoothed[1], self._smoothed[2]],
            dtype=np.float32,
        )

        total_raw  = 0.0
        last_obs_rgb = None
        last_info    = {}
        terminated = truncated = False

        for _ in range(ACTION_REPEAT):
            obs_rgb, raw_r, terminated, truncated, info = self.env.step(full_action)
            total_raw   += float(raw_r)
            last_obs_rgb = obs_rgb
            last_info    = info
            if terminated or truncated:
                break

        self._raw_ep_rew += total_raw
        self._step_cnt   += 1

        speed = self._get_speed()
        obs_vec, rays, on_road, curvature, fa, la, ra = \
            self._build_obs(last_obs_rgb, speed)
        self._prev_speed = speed

        r   = total_raw
        _rb = {}

        # 1. Tile bonus — only when new_tiles > 0
        tiles     = last_info.get("tile_visited_count", self._prev_tiles)
        new_tiles = max(0, tiles - self._prev_tiles)
        _rb["tile_bonus"]         = new_tiles * TILE_BONUS
        r                        += _rb["tile_bonus"]
        self._tile_bonus_total   += _rb["tile_bonus"]
        self._prev_tiles          = tiles

        # 2. Steering guide
        steering_guide         = float(self._smoothed[0]) * (ra - la)
        _rb["steering_guide"]  = steering_guide * STEERING_GUIDE
        r                     += _rb["steering_guide"]

        # 3. On-road speed bonus
        _rb["speed_bonus"] = speed * SPEED_BONUS if on_road else 0.0
        r                 += _rb["speed_bonus"]

        # 4. Curve braking
        _rb["curve_brake"] = (
            curvature * CURVE_BRAKE_BON
            if (on_road and curvature > 0.3 and self._smoothed[1] < 0.4) else 0.0
        )
        r += _rb["curve_brake"]

        # 5. Front danger
        _rb["front_danger"] = -FRONT_DANGER_PEN if fa < 0.10 else 0.0
        r                  += _rb["front_danger"]

        # 6. Steer jitter penalty
        steer_jitter      = abs(float(self._smoothed[0]) - self._prev_smoothed_steer)
        _rb["jitter_pen"] = -steer_jitter * STEER_PEN
        r                += _rb["jitter_pen"]

        # 7. Anti-spin
        _rb["anti_spin"] = (
            -ANTI_SPIN_PEN
            if (abs(float(self._smoothed[0])) > 0.7 and speed < 0.02) else 0.0
        )
        r += _rb["anti_spin"]

        # 8. Offroad
        if on_road:
            self._offroad_cnt   = 0
            _rb["offroad_pen"]  = 0.0
            _rb["gas_offroad"]  = 0.0
        else:
            self._offroad_cnt  += 1
            _rb["offroad_pen"]  = -OFFROAD_PEN
            _rb["gas_offroad"]  = -float(self._smoothed[1]) * GAS_OFFROAD_PEN
        r += _rb["offroad_pen"] + _rb["gas_offroad"]

        # 9. Still detection (warmup 50 steps)
        if self._step_cnt > 50 and speed < STILL_SPEED_THR:
            self._still_cnt += 1
        else:
            self._still_cnt = 0

        # 10. Termination conditions
        _rb["term_pen"] = 0.0
        if self._offroad_cnt >= MAX_OFFROAD:
            _rb["term_pen"] -= 5.0
            terminated = True
            last_info["term_reason"] = "offroad_timeout"
        if self._still_cnt >= MAX_STILL:
            _rb["term_pen"] -= 2.0
            terminated = True
            last_info["term_reason"] = "still_timeout"
        r += _rb["term_pen"]

        last_info.update(dict(
            raw_reward       = float(total_raw),
            shaped_reward    = float(r),
            raw_ep_reward    = self._raw_ep_rew,
            reward_breakdown = _rb,
            on_road          = on_road,
            offroad_cnt      = self._offroad_cnt,
            speed            = speed,
            curvature        = curvature,
            front_avg        = fa,
            tiles            = tiles,
            tile_bonus_total = self._tile_bonus_total,
            tile_bonus_mean  = (self._tile_bonus_total / self._step_cnt
                                if self._step_cnt > 0 else 0.0),
        ))
        return obs_vec, r, terminated, truncated, last_info


print("✅ Cell 6 — CarRacingRayWrapper defined")


In [ ]:
# Cell 7 — VecEnv builders

from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecNormalize
from stable_baselines3.common.monitor import Monitor


def make_train_env(rank: int = 0):
    def _init():
        env = gym.make(CAR_RACING_ENV_ID, continuous=True, domain_randomize=False)
        env = CarRacingRayWrapper(env)
        return Monitor(env)
    return _init


def make_eval_env(seed: int = 999):
    def _init():
        env = gym.make(CAR_RACING_ENV_ID, render_mode="rgb_array",
                       continuous=True, domain_randomize=False)
        env = CarRacingRayWrapper(env)
        env.reset(seed=seed)
        return Monitor(env)
    return _init


def build_train_vec():
    vec = DummyVecEnv([make_train_env(i) for i in range(N_ENVS)])
    vec = VecFrameStack(vec, n_stack=N_STACK)
    vec = VecNormalize(vec, norm_obs=True, norm_reward=True,
                       clip_obs=10.0, clip_reward=10.0, gamma=GAMMA)
    return vec


def build_eval_vec(seed: int = 999):
    vec = DummyVecEnv([make_eval_env(seed=seed)])
    vec = VecFrameStack(vec, n_stack=N_STACK)
    vec = VecNormalize(vec, norm_obs=True, norm_reward=False,
                       training=False, clip_obs=10.0, gamma=GAMMA)
    return vec


def build_eval_vec_from_disk(norm_path: str, seed: int = 0):
    """Build fresh eval stack and load saved VecNormalize stats from disk."""
    vec = DummyVecEnv([make_eval_env(seed=seed)])
    vec = VecFrameStack(vec, n_stack=N_STACK)
    vec = VecNormalize.load(norm_path, vec)
    vec.training    = False
    vec.norm_reward = False
    return vec


print("✅ Cell 7 — VecEnv builders defined")


In [ ]:
# Cell 8 — Callback and artifact helpers

from stable_baselines3.common.callbacks import (
    BaseCallback, CheckpointCallback, EvalCallback,
)
from stable_baselines3.common.vec_env import VecNormalize as _VN


class SyncedEvalCallback(EvalCallback):
    """
    Before each eval: sync train VecNormalize obs_rms -> eval env (fixes normalization sync issue).
    After new best: save best_vecnormalize.pkl and back up evaluations.npz to Drive.
    """

    @staticmethod
    def _find_vn(env):
        while env is not None and not isinstance(env, _VN):
            env = getattr(env, "venv", None)
        return env

    def _on_step(self) -> bool:
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            train_vn = self._find_vn(self.training_env)
            eval_vn  = self._find_vn(self.eval_env)
            if train_vn and eval_vn:
                try:
                    from stable_baselines3.common.vec_env import sync_envs_normalization
                    sync_envs_normalization(train_vn, eval_vn)
                except ImportError:
                    import copy
                    eval_vn.obs_rms = copy.deepcopy(train_vn.obs_rms)
                    eval_vn.ret_rms = copy.deepcopy(train_vn.ret_rms)

        old_best = self.best_mean_reward
        result   = super()._on_step()

        if self.best_model_save_path and self.best_mean_reward > old_best:
            # Save best VecNormalize paired with best model
            train_vn = self._find_vn(self.training_env)
            if train_vn:
                best_norm = os.path.join(self.best_model_save_path, "best_vecnormalize.pkl")
                train_vn.save(best_norm)
                _backup(best_norm)

            # Back up evaluations.npz to Drive on every improvement
            eval_npz = os.path.join(self.best_model_save_path, "evaluations.npz")
            if os.path.isfile(eval_npz):
                bck = os.path.join(EVAL_BCK_DIR,
                                   f"evaluations_{self.num_timesteps}.npz")
                shutil.copy2(eval_npz, bck)
                _backup(eval_npz)

        return result


class SaveVecNormCallback(CheckpointCallback):
    """Saves VecNormalize pkl alongside each checkpoint + Drive backup."""

    def _on_step(self) -> bool:
        if self.n_calls % self.save_freq == 0:
            norm_p = os.path.join(
                self.save_path,
                f"vecnorm_{self.num_timesteps}_steps.pkl",
            )
            self.training_env.save(norm_p)
            _backup(norm_p)
        result = super()._on_step()
        model_p = os.path.join(
            self.save_path,
            f"{self.name_prefix}_{self.num_timesteps}_steps.zip",
        )
        if os.path.isfile(model_p):
            _backup(model_p)
        return result


class EvalLiveLogger(BaseCallback):
    """Append one CSV row per training step batch to eval_live.csv."""

    def __init__(self, csv_path: str, verbose=0):
        super().__init__(verbose)
        self._csv = csv_path
        if not os.path.isfile(csv_path):
            with open(csv_path, "w") as f:
                f.write("timestep,mean_reward,std_reward\n")

    def _on_step(self) -> bool:
        ep_rew = self.locals.get("rewards")
        if ep_rew is not None and len(ep_rew):
            with open(self._csv, "a") as f:
                f.write(f"{self.num_timesteps},"
                        f"{float(np.mean(ep_rew)):.4f},"
                        f"{float(np.std(ep_rew)):.4f}\n")
        return True


class RawRewardLogger(BaseCallback):
    def __init__(self, log_freq=10_000, verbose=1):
        super().__init__(verbose)
        self.log_freq = log_freq

    def _on_step(self) -> bool:
        if self.n_calls % self.log_freq == 0:
            raws = [i.get("raw_ep_reward") for i in self.locals.get("infos", [])
                    if i.get("raw_ep_reward") is not None]
            if raws and self.verbose:
                print(f"  [RAW] step={self.num_timesteps:>9,}  "
                      f"raw_ep_rew={np.mean(raws):.1f}")
        return True


class NaNGuard(BaseCallback):
    def _on_step(self) -> bool:
        loss = self.logger.name_to_value.get("train/loss")
        if loss is not None and not np.isfinite(loss):
            print(f"  [STOP] NaN loss at step {self.num_timesteps}")
            return False
        return True


def make_callbacks(train_vec, eval_vec):
    ckpt_cb = SaveVecNormCallback(
        save_freq=CKPT_FREQ, save_path=CKPT_DIR, name_prefix="final", verbose=0,
    )
    eval_cb = SyncedEvalCallback(
        eval_env=eval_vec,
        eval_freq=EVAL_FREQ,
        n_eval_episodes=N_EVAL_EPS,
        best_model_save_path=BEST_DIR,
        log_path=BEST_DIR,
        deterministic=True,
        verbose=1,
    )
    live_cb = EvalLiveLogger(csv_path=f"{LOG_DIR}/eval_live.csv")
    raw_cb  = RawRewardLogger(log_freq=10_000)
    nan_cb  = NaNGuard()
    return [ckpt_cb, eval_cb, live_cb, raw_cb, nan_cb]


def checkpoint_inventory():
    """Write checkpoint_inventory.csv and return DataFrame."""
    import glob as _glob
    rows = []
    for z in sorted(_glob.glob(f"{CKPT_DIR}/*.zip")):
        stem = os.path.basename(z).replace("final_", "").replace("_steps.zip", "")
        try:
            steps = int(stem)
        except ValueError:
            steps = -1
        norm = z.replace("final_", "vecnorm_").replace(".zip", ".pkl")
        rows.append({
            "model_zip": z,
            "steps":     steps,
            "norm_pkl":  norm if os.path.isfile(norm) else "",
            "size_kb":   round(_kb(z), 1),
        })
    df  = pd.DataFrame(rows)
    csv = f"{TABLE_DIR}/checkpoint_inventory.csv"
    df.to_csv(csv, index=False)
    _backup(csv)
    return df


print("✅ Cell 8 — callbacks and artifact helpers defined")


In [ ]:
# Cell 9 — Evaluation helpers Corrected
# Purpose:
#   Fix post-training evaluation seed isolation.
#   The old evaluator produced identical first-frame hashes and identical rewards
#   for all seeds, which means COMPARE5 / RANDOM10 were invalid.
#
# Key fixes:
#   1. Force seed into VecEnv before reset.
#   2. Hash raw rendered RGB frame, not normalized feature observation.
#   3. Add raw / wrapped / VecNormalize seed-diversity diagnostics.
#   4. Fail loudly if first-frame hashes are duplicated.
#   5. Print artifact sanity so we know which model/norm pair is being evaluated.
#
# IMPORTANT:
#   This cell does NOT train.
#   This cell does NOT call model.learn().
#   After running this cell, rerun Cell 20 first.

import os
import time
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import gymnasium as gym

from stable_baselines3 import PPO as _PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecNormalize
from stable_baselines3.common.monitor import Monitor

from IPython.display import display, Image as IPyImage


def _hash_array(arr, n=12):
    """Return a short stable hash for a numpy-compatible array."""
    arr = np.asarray(arr)
    return hashlib.md5(arr.tobytes()).hexdigest()[:n]


def _env_id():
    """Resolve the CarRacing env id from notebook globals."""
    return globals().get("CAR_RACING_ENV_ID", globals().get("ENV_ID", "CarRacing-v3"))


def _wrapper_class():
    """Resolve the custom CarRacing wrapper class from notebook globals."""
    if "CarRacingRayWrapper" in globals():
        return CarRacingRayWrapper
    if "CarRacingRayWrapper" in globals():
        return CarRacingRayWrapper
    raise NameError(
        "Cannot find CarRacingRayWrapper in notebook globals. "
        "Run the wrapper-definition cell before Cell 9."
    )


def _make_single_eval_env():
    """
    Build one unseeded eval env.
    Seeding is applied later through VecEnv.seed(seed) immediately before reset.
    """
    raw = gym.make(
        _env_id(),
        continuous=True,
        render_mode="rgb_array",
        domain_randomize=False,
    )
    wrapped = _wrapper_class()(raw)
    return Monitor(wrapped)


def _unwrap_to_dummy_vecenv(vec):
    """
    Unwrap VecNormalize / VecFrameStack down to DummyVecEnv.

    Expected chain:
        VecNormalize -> VecFrameStack -> DummyVecEnv

    Returns:
        inner DummyVecEnv
    """
    cur = vec
    seen = []

    while hasattr(cur, "venv"):
        seen.append(type(cur).__name__)
        cur = cur.venv

    if not hasattr(cur, "envs"):
        raise TypeError(
            f"Could not unwrap to DummyVecEnv. "
            f"Wrapper chain={seen}, final={type(cur).__name__}"
        )

    return cur


def _render_rgb_from_vec(vec):
    """
    Render the current raw RGB frame from the underlying env after vec.reset().

    This is used for seed verification.
    Do NOT hash the normalized feature observation; that can be identical even
    when the underlying track differs.
    """
    dummy = _unwrap_to_dummy_vecenv(vec)
    env0 = dummy.envs[0]

    frame = env0.render()

    if frame is None:
        try:
            frame = env0.unwrapped.render()
        except Exception:
            frame = None

    if frame is None:
        raise RuntimeError(
            "Could not render raw RGB frame from eval env. "
            "Check render_mode='rgb_array' in eval env construction."
        )

    return np.asarray(frame)


def _raw_seed_hash(seed):
    """
    Direct raw Gymnasium CarRacing-v3 seed test.

    Different seeds should usually produce different first-frame hashes.
    If this layer duplicates, the issue is below our wrapper.
    """
    env = gym.make(
        _env_id(),
        continuous=True,
        render_mode="rgb_array",
        domain_randomize=False,
    )

    try:
        env.reset(seed=int(seed))
        frame = env.render()
        return _hash_array(frame)
    finally:
        env.close()


def _wrapped_seed_hash(seed):
    """
    Wrapped env seed test before VecNormalize.

    This checks whether the custom wrapper preserves seed diversity.
    """
    env = _make_single_eval_env()

    try:
        env.reset(seed=int(seed))
        frame = env.render()
        return _hash_array(frame)
    finally:
        env.close()


def build_seeded_eval_vec_from_disk(norm_path: str, seed: int):
    """
    Fresh eval VecEnv for exactly one seed.

    Critical:
        - build a fresh env for every seed
        - call VecEnv.seed(seed)
        - reset only after seed is set
        - load VecNormalize from disk
        - set training=False and norm_reward=False for evaluation
    """
    seed = int(seed)

    base = DummyVecEnv([lambda: _make_single_eval_env()])
    base.seed(seed)

    base = VecFrameStack(base, n_stack=N_STACK)

    vec = VecNormalize.load(norm_path, base)
    vec.training = False
    vec.norm_reward = False

    # SB3 VecEnv.seed(seed) applies on next reset.
    # Call again after VecNormalize wraps the env to avoid losing the seed.
    vec.seed(seed)

    return vec


def _vec_seed_hash(norm_path, seed):
    """
    VecNormalize-loaded eval env seed test.

    If raw/wrapped hashes differ but this duplicates, the bug is in
    VecEnv / VecNormalize seeding/reset path.
    """
    vec = build_seeded_eval_vec_from_disk(norm_path, seed)

    try:
        _ = vec.reset()
        frame = _render_rgb_from_vec(vec)
        return _hash_array(frame)
    finally:
        vec.close()


def seed_diversity_diagnostic(seeds, norm_path=None, strict=True):
    """
    Three-layer seed check:
        1. raw Gymnasium env
        2. custom wrapped env
        3. VecNormalize-loaded eval env, if norm_path exists

    If hashes duplicate at any layer, evaluation is not valid.
    """
    seeds = [int(s) for s in seeds]

    rows = []

    for s in seeds:
        rows.append({
            "layer": "raw_gym",
            "seed": s,
            "hash": _raw_seed_hash(s),
        })

    for s in seeds:
        rows.append({
            "layer": "wrapped",
            "seed": s,
            "hash": _wrapped_seed_hash(s),
        })

    if norm_path is not None and os.path.isfile(norm_path):
        for s in seeds:
            rows.append({
                "layer": "vecnorm",
                "seed": s,
                "hash": _vec_seed_hash(norm_path, s),
            })

    df = pd.DataFrame(rows)

    print("\n── Seed Diversity Diagnostic ──")
    print(df.to_string(index=False))

    bad_layers = []
    for layer, g in df.groupby("layer"):
        if g["hash"].nunique() < len(g):
            bad_layers.append(layer)

    if bad_layers:
        msg = f"FATAL: duplicate first-frame hashes in layers: {bad_layers}"
        if strict:
            raise RuntimeError(msg)
        print("WARNING:", msg)
    else:
        print("✅ Seed diversity confirmed in all tested layers.")

    return df


def _show_png(path):
    """Display a saved PNG in Colab if it exists."""
    if os.path.isfile(path):
        display(IPyImage(filename=path))
    else:
        print(f"WARNING: plot not found: {path}")


def _resolve_model_norm(model_path=None, norm_path=None, prefer="best"):
    """
    Return (model_path_no_ext, norm_path).

    prefer="best":
        Use best model/norm first, then final fallback.

    prefer="final":
        Use final model/norm first, then best fallback.
    """
    best_model = f"{BEST_DIR}/best_model"
    best_norm = f"{BEST_DIR}/best_vecnormalize.pkl"

    final_model = f"{CKPT_DIR}/model_final"
    final_norm = f"{CKPT_DIR}/vecnorm_final.pkl"

    if model_path is None or norm_path is None:
        if prefer == "final":
            model_path = final_model if os.path.isfile(final_model + ".zip") else best_model
            norm_path = final_norm if os.path.isfile(final_norm) else best_norm
        else:
            model_path = best_model if os.path.isfile(best_model + ".zip") else final_model
            norm_path = best_norm if os.path.isfile(best_norm) else final_norm

    return model_path, norm_path


def artifact_sanity(model_path=None, norm_path=None, prefer="best"):
    """
    Print artifact pair info and EvalCallback npz best score.

    This answers:
        Are we evaluating the same artifact pair that reached the high eval score?
    """
    model_path, norm_path = _resolve_model_norm(
        model_path=model_path,
        norm_path=norm_path,
        prefer=prefer,
    )

    print("\n── Artifact Sanity ──")
    print(f"Model path : {model_path}.zip")
    print(f"Norm path  : {norm_path}")

    for p in [model_path + ".zip", norm_path]:
        if os.path.isfile(p):
            print(
                f"OK   {p}  "
                f"size={os.path.getsize(p) / 1024:.1f} KB  "
                f"mtime={time.ctime(os.path.getmtime(p))}"
            )
        else:
            print(f"MISS {p}")

    eval_npz = f"{BEST_DIR}/evaluations.npz"

    if os.path.isfile(eval_npz):
        try:
            data = np.load(eval_npz)
            timesteps = data.get("timesteps", None)
            results = data.get("results", None)

            if timesteps is not None and results is not None:
                means = results.mean(axis=1)
                best_i = int(np.argmax(means))
                print(
                    f"evaluations.npz best: "
                    f"step={int(timesteps[best_i])}, "
                    f"mean={means[best_i]:.2f}"
                )
                print(f"evaluations.npz last means: {np.round(means[-5:], 2)}")
            else:
                print(f"evaluations.npz keys: {list(data.keys())}")

        except Exception as exc:
            print(f"WARNING: could not read evaluations.npz: {exc}")
    else:
        print(f"WARNING: missing {eval_npz}")

    return model_path, norm_path


def _run_eval_episode(model_path: str, norm_path: str, seed: int,
                      seed_type: str, max_steps: int = 1500) -> dict:
    """
    One deterministic eval episode with a fresh seeded env stack loaded from disk.

    Important:
        first_frame_hash is computed from raw rendered RGB after seeded reset.
    """
    seed = int(seed)

    vec = build_seeded_eval_vec_from_disk(norm_path, seed=seed)
    model = _PPO.load(model_path, env=vec)

    obs = vec.reset()
    ff_hash = _hash_array(_render_rgb_from_vec(vec))

    steer_log, gas_log, brake_log = [], [], []
    raw_rew = 0.0
    shaped_rew = 0.0
    tile_bonus_total = 0.0
    frames = 0
    done_reason = "max_steps"

    for _ in range(max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, infos = vec.step(action)
        info = infos[0]

        raw_rew += float(info.get("raw_reward", 0.0))
        shaped_rew += float(info.get("shaped_reward", 0.0))

        reward_breakdown = info.get("reward_breakdown", {}) or {}
        tile_bonus_total += float(reward_breakdown.get("tile_bonus", 0.0))

        action_flat = np.asarray(action).reshape(-1)
        steer_log.append(float(action_flat[0]))
        gas_log.append(float(action_flat[1]))
        brake_log.append(float(action_flat[2]))

        frames += 1

        if bool(done[0]):
            done_reason = info.get("term_reason", info.get("done_reason", "env_done"))
            break

    vec.close()

    steer = np.asarray(steer_log, dtype=float)
    gas = np.asarray(gas_log, dtype=float)
    brake = np.asarray(brake_log, dtype=float)

    return {
        "seed": seed,
        "seed_type": seed_type,
        "raw_reward": round(raw_rew, 2),
        "shaped_reward": round(shaped_rew, 2),
        "episode_frames": frames,
        "done_reason": done_reason,
        "steer_delta_mean": round(
            float(np.mean(np.abs(np.diff(steer)))) if len(steer) > 1 else 0.0,
            4,
        ),
        "steer_abs_mean": round(float(np.mean(np.abs(steer))) if len(steer) else 0.0, 4),
        "gas_mean": round(float(np.mean(gas)) if len(gas) else 0.0, 4),
        "brake_mean": round(float(np.mean(brake)) if len(brake) else 0.0, 4),
        "first_frame_hash": ff_hash,
        "tile_bonus_total": round(tile_bonus_total, 2),
        "tile_bonus_mean": round(tile_bonus_total / max(frames, 1), 4),
    }


def _assert_unique_hashes(rows, label):
    """Hard fail if any first-frame hashes are duplicated."""
    hashes = [r["first_frame_hash"] for r in rows]

    if len(set(hashes)) < len(hashes):
        df = pd.DataFrame(rows)
        print(df[[
            "seed",
            "raw_reward",
            "episode_frames",
            "done_reason",
            "first_frame_hash",
        ]].to_string(index=False))

        raise RuntimeError(
            f"{label}: duplicate first-frame hashes detected. "
            "Seed isolation is still broken; do not use these evaluation results."
        )

    print(f"✅ {label}: all first-frame hashes unique.")


def record_episode_fixed(model_path: str, norm_path: str, seed: int,
                         max_steps: int = 1500):
    """
    Fixed video recorder using the corrected seeded eval path.

    Returns:
        frames, metadata
    """
    seed = int(seed)

    vec = build_seeded_eval_vec_from_disk(norm_path, seed=seed)
    model = _PPO.load(model_path, env=vec)

    obs = vec.reset()
    first_hash = _hash_array(_render_rgb_from_vec(vec))

    frames = []
    raw_rew = 0.0
    shaped_rew = 0.0
    done_reason = "max_steps"

    for _ in range(max_steps):
        frames.append(_render_rgb_from_vec(vec))

        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, infos = vec.step(action)
        info = infos[0]

        raw_rew += float(info.get("raw_reward", 0.0))
        shaped_rew += float(info.get("shaped_reward", 0.0))

        if bool(done[0]):
            done_reason = info.get("term_reason", info.get("done_reason", "env_done"))
            frames.append(_render_rgb_from_vec(vec))
            break

    vec.close()

    metadata = {
        "seed": seed,
        "raw_reward": round(raw_rew, 2),
        "shaped_reward": round(shaped_rew, 2),
        "episode_frames": len(frames),
        "done_reason": done_reason,
        "first_frame_hash": first_hash,
        "source_fps": 50,
    }

    return frames, metadata


print("✅ Cell 9 Corrected — seeded evaluation helpers defined")



In [ ]:
# Cell 9B — Reload-safe evaluation: avoid PPO.load(..., env=vec) resetting eval seed

# Run this AFTER patched Cell 9, then rerun Cell 20.

# Root cause found:
# seed_diversity_diagnostic() confirmed raw/wrapped/vecnorm hashes are unique.
# But _run_eval_episode() still collapsed all seeds to seed 42.
# Therefore PPO.load(model_path, env=vec) is likely reseeding the env from the
# stored model seed.

# Fix:
# Load PPO policy WITHOUT env.
# Build and seed the eval VecNormalize env AFTER model loading.
# Use model.predict(obs, deterministic=True) without attaching env to model.

def _load_policy_only(model_path: str):
    """Load PPO model without attaching an env, to avoid env reseeding side effects."""
    return _PPO.load(model_path)

def _run_eval_episode(model_path: str, norm_path: str, seed: int,
                      seed_type: str, max_steps: int = 1500) -> dict:
    """
    One deterministic eval episode with a fresh seeded env stack loaded from disk.

    Important:
    - PPO is loaded WITHOUT env to avoid reseeding the eval env.
    - first_frame_hash is computed from raw rendered RGB after seeded reset.
    """
    seed = int(seed)

    model = _load_policy_only(model_path)
    vec = build_seeded_eval_vec_from_disk(norm_path, seed=seed)

    obs = vec.reset()
    ff_hash = _hash_array(_render_rgb_from_vec(vec))

    steer_log, gas_log, brake_log = [], [], []
    raw_rew = 0.0
    shaped_rew = 0.0
    tile_bonus_total = 0.0
    frames = 0
    done_reason = "max_steps"

    for _ in range(max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, infos = vec.step(action)
        info = infos[0]

        raw_rew += float(info.get("raw_reward", 0.0))
        shaped_rew += float(info.get("shaped_reward", 0.0))

        reward_breakdown = info.get("reward_breakdown", {}) or {}
        tile_bonus_total += float(reward_breakdown.get("tile_bonus", 0.0))

        action_flat = np.asarray(action).reshape(-1)
        steer_log.append(float(action_flat[0]))
        gas_log.append(float(action_flat[1]))
        brake_log.append(float(action_flat[2]))

        frames += 1

        if bool(done[0]):
            done_reason = info.get("term_reason", info.get("done_reason", "env_done"))
            break

    vec.close()

    steer = np.asarray(steer_log, dtype=float)
    gas = np.asarray(gas_log, dtype=float)
    brake = np.asarray(brake_log, dtype=float)

    return {
        "seed": seed,
        "seed_type": seed_type,
        "raw_reward": round(raw_rew, 2),
        "shaped_reward": round(shaped_rew, 2),
        "episode_frames": frames,
        "done_reason": done_reason,
        "steer_delta_mean": round(
            float(np.mean(np.abs(np.diff(steer)))) if len(steer) > 1 else 0.0,
            4,
        ),
        "steer_abs_mean": round(float(np.mean(np.abs(steer))) if len(steer) else 0.0, 4),
        "gas_mean": round(float(np.mean(gas)) if len(gas) else 0.0, 4),
        "brake_mean": round(float(np.mean(brake)) if len(brake) else 0.0, 4),
        "first_frame_hash": ff_hash,
        "tile_bonus_total": round(tile_bonus_total, 2),
        "tile_bonus_mean": round(tile_bonus_total / max(frames, 1), 4),
    }

def record_episode_fixed(model_path: str, norm_path: str, seed: int,
                          max_steps: int = 1500):
    """
    Fixed video recorder using corrected seeded eval path.

    Important:
    - PPO is loaded WITHOUT env to avoid reseeding the eval env.
    """
    seed = int(seed)

    model = _load_policy_only(model_path)
    vec = build_seeded_eval_vec_from_disk(norm_path, seed=seed)

    obs = vec.reset()
    first_hash = _hash_array(_render_rgb_from_vec(vec))

    frames = []
    raw_rew = 0.0
    shaped_rew = 0.0
    done_reason = "max_steps"

    for _ in range(max_steps):
        frames.append(_render_rgb_from_vec(vec))

        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, infos = vec.step(action)
        info = infos[0]

        raw_rew += float(info.get("raw_reward", 0.0))
        shaped_rew += float(info.get("shaped_reward", 0.0))

        if bool(done[0]):
            done_reason = info.get("term_reason", info.get("done_reason", "env_done"))
            frames.append(_render_rgb_from_vec(vec))
            break

    vec.close()

    metadata = {
        "seed": seed,
        "raw_reward": round(raw_rew, 2),
        "shaped_reward": round(shaped_rew, 2),
        "episode_frames": len(frames),
        "done_reason": done_reason,
        "first_frame_hash": first_hash,
        "source_fps": 50,
    }

    return frames, metadata

print("✅ Cell 9B PATCH applied — PPO is now loaded without env during eval/video")

In [ ]:
# Cell 10 — Video export helpers

import base64
from IPython.display import HTML, display as _ipy_display
from stable_baselines3 import PPO as _PPO


def record_episode(model_path: str, norm_path: str, seed: int,
                   max_steps: int = 1500) -> list:
    """Record one episode from disk-loaded model+norm. Returns list of RGB frames."""
    vec = build_eval_vec_from_disk(norm_path, seed=seed)
    m   = _PPO.load(model_path, env=vec)
    obs = vec.reset()
    frames = []

    for step in range(max_steps):
        action, _ = m.predict(obs, deterministic=True)
        obs, _, done, _ = vec.step(action)
        if step % 2 == 0:
            frame = vec.render()
            if frame is not None:
                if isinstance(frame, list):
                    frame = frame[0]
                if isinstance(frame, np.ndarray):
                    frames.append(frame)
        if done[0]:
            break

    vec.close()
    return frames


def save_video(frames: list, path: str, fps: int = 25) -> str:
    if not frames:
        print("WARNING: No frames to save.")
        return ""
    imageio.mimsave(path, frames, fps=fps)
    _backup(path)
    return path


def show_video(path: str):
    if not os.path.isfile(path):
        print(f"WARNING: Video not found: {path}")
        return
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    _ipy_display(HTML(
        f'<video width="480" controls autoplay loop>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f'</video>'
    ))


print("✅ Cell 10 — video helpers defined")


In [ ]:
# Cell 11 — No-training smoke test: raw CarRacing-v3
# Verifies gymnasium install and env API. NO model.learn().

try:
    _e = gym.make(CAR_RACING_ENV_ID, render_mode="rgb_array",
                  continuous=True, domain_randomize=False)
except Exception as exc:
    raise RuntimeError(
        f"CarRacing-v3 init failed: {exc}. "
        "Check gymnasium[box2d] and swig — re-run Cell 1 and restart runtime."
    ) from exc

_obs, _ = _e.reset(seed=0)
assert _obs.shape == (96, 96, 3), f"Unexpected obs shape: {_obs.shape}"

_out = _e.step(np.array([0.0, 0.5, 0.0], dtype=np.float32))
assert len(_out) == 5, f"Expected 5-tuple from step(), got {len(_out)}"

_e.close()
del _e, _obs, _out

print(f"✅ Cell 11 — raw {CAR_RACING_ENV_ID} OK  obs=(96,96,3)  action=3D")


In [ ]:
# Cell 12 — No-training smoke test: wrapper obs/action/reward
# Verifies CarRacingRayWrapper shape, reward fields, tile_bonus gate. NO model.learn().

from stable_baselines3.common.env_checker import check_env

_base = gym.make(CAR_RACING_ENV_ID, continuous=True, domain_randomize=False)
_w    = CarRacingRayWrapper(_base)

check_env(_w, warn=True)

_obs, _info = _w.reset(seed=SEED)
assert _obs.shape == (14,), f"obs shape {_obs.shape} != (14,)"
assert np.all(np.isfinite(_obs)), "NaN/Inf in reset obs"

for _i in range(20):
    _a = _w.action_space.sample()
    _obs, _rew, _term, _trunc, _info = _w.step(_a)
    assert _obs.shape  == (14,),  f"step {_i}: obs {_obs.shape}"
    assert np.all(np.isfinite(_obs)), f"step {_i}: NaN/Inf in obs"
    assert np.isfinite(_rew),        f"step {_i}: reward {_rew}"
    assert "raw_reward"       in _info
    assert "tile_bonus_total" in _info
    assert "tile_bonus_mean"  in _info
    if _term or _trunc:
        _obs, _info = _w.reset()

_w.close()
del _base, _w

print("✅ Cell 12 — wrapper OK  obs=(14,)  action=(3,)  tile_bonus gated")


In [ ]:
# Cell 13 — No-training smoke test: VecEnv + VecNormalize shape
# Builds and immediately closes train/eval VecEnv. NO model.learn().

_tv = build_train_vec()
_ev = build_eval_vec(seed=999)

_obs = _tv.reset()
assert _obs.shape == (N_ENVS, 14 * N_STACK), \
    f"train obs {_obs.shape} != ({N_ENVS}, {14*N_STACK})"
assert np.all(np.isfinite(_obs)), "NaN/Inf in train_vec reset"

_ev.reset()

_tv.close()
_ev.close()
del _tv, _ev, _obs

print(f"✅ Cell 13 — VecEnv OK  train=({N_ENVS},{14*N_STACK})  eval=(1,{14*N_STACK})")


In [ ]:
# Cell 14 — No-training smoke test: PPO build and predict only
# Builds train_vec, eval_vec, and model for training.
# Verifies predict() shape and finite outputs. NO model.learn().

from stable_baselines3 import PPO

train_vec = build_train_vec()
eval_vec  = build_eval_vec(seed=999)

model = PPO(
    "MlpPolicy",
    train_vec,
    learning_rate = lr_schedule,
    n_steps       = N_STEPS,
    batch_size    = BATCH_SIZE,
    n_epochs      = N_EPOCHS,
    gamma         = GAMMA,
    gae_lambda    = GAE_LAMBDA,
    clip_range    = clip_schedule,
    ent_coef      = ENT_COEF,
    vf_coef       = VF_COEF,
    max_grad_norm = MAX_GRAD_NORM,
    policy_kwargs = dict(net_arch=[256, 256]),
    verbose       = 1,
    seed          = SEED,
    device        = DEVICE,
)

_params = sum(p.numel() for p in model.policy.parameters())
print(f"Policy   : MlpPolicy [256,256]  params={_params:,}  device={DEVICE}")

_obs = train_vec.reset()
_act, _ = model.predict(_obs, deterministic=True)
assert _act.shape == (N_ENVS, 3), f"action shape {_act.shape}"
assert np.all(np.isfinite(_act)), "NaN/Inf in predicted action"

_obs2, _rew, _, _ = train_vec.step(_act)
assert np.all(np.isfinite(_obs2)), "NaN/Inf in obs after step"
assert np.all(np.isfinite(_rew)),  "NaN/Inf in reward after step"
del _obs, _act, _obs2, _rew

print(f"USE_SDE       = {USE_SDE}")
print(f"LR            = {INIT_LR:.2e} -> {FINAL_LR:.2e}")
print(f"ENT_COEF      = {ENT_COEF}")
print("Model weights untouched — smoke test only.")
print("✅ Cell 14 — PPO build+predict OK")
print("   Next: Cell 15 (safety stop) -> Cell 16 (training)")


In [ ]:
# Cell 15 — Required STOP before training
# ─────────────────────────────────────────────────────────────────────
# Cells 0-14 are complete. All smoke tests should have passed.
#
# To start 500k training:
#   Option A: edit Cell 3 — set DRY_RUN=False, TRAIN_APPROVED=True
#   Option B: uncomment the two lines below and re-run this cell,
#             then run Cell 16.
# ─────────────────────────────────────────────────────────────────────
# DRY_RUN        = False   # uncomment when ready
# TRAIN_APPROVED = True    # uncomment when ready

print("=" * 60)
print("STOP — All smoke tests passed.")
print(f"  DRY_RUN        = {DRY_RUN}")
print(f"  TRAIN_APPROVED = {TRAIN_APPROVED}")
if DRY_RUN or not TRAIN_APPROVED:
    print()
    print("  Cell 16 will REFUSE to train with current settings.")
    print("  Set DRY_RUN=False and TRAIN_APPROVED=True to proceed.")
else:
    print()
    print("  Guards cleared. Running Cell 16 will start 500k training.")
print("=" * 60)


In [ ]:
# Cell 16 — Main 500k scratch training
# Requires: DRY_RUN=False AND TRAIN_APPROVED=True (set in Cell 3 or Cell 15)

assert not DRY_RUN and TRAIN_APPROVED, (
    "Training blocked.\n"
    "  Set DRY_RUN=False and TRAIN_APPROVED=True in Cell 3 (or Cell 15), "
    "then re-run this cell."
)

FINAL_MODEL = f"{CKPT_DIR}/model_final"
FINAL_NORM  = f"{CKPT_DIR}/vecnorm_final.pkl"

_callback = make_callbacks(train_vec, eval_vec)

print(f"Training {TOTAL_TIMESTEPS:,} steps  |  "
      f"eval every {EVAL_FREQ*N_ENVS:,}  |  "
      f"ckpt every {CKPT_FREQ*N_ENVS:,}")
print(f"BEST_DIR  : {BEST_DIR}")
print(f"DRIVE_OK  : {_DRIVE_OK}")
_t0 = time.time()

try:
    model.learn(
        total_timesteps  = TOTAL_TIMESTEPS,
        reset_num_timesteps = True,
        progress_bar     = True,
        callback         = _callback,
    )
except KeyboardInterrupt:
    print("\n[INTERRUPTED] Saving emergency checkpoint...")
    model.save(f"{CKPT_DIR}/model_interrupt")
    train_vec.save(f"{CKPT_DIR}/vecnorm_interrupt.pkl")
except Exception as _exc:
    model.save(f"{CKPT_DIR}/model_emergency")
    train_vec.save(f"{CKPT_DIR}/vecnorm_emergency.pkl")
    raise

_wall = time.time() - _t0

model.save(FINAL_MODEL)
train_vec.save(FINAL_NORM)
_backup(FINAL_MODEL + ".zip")
_backup(FINAL_NORM)

_best_zip  = f"{BEST_DIR}/best_model.zip"
_best_norm = f"{BEST_DIR}/best_vecnormalize.pkl"
_eval_npz  = f"{BEST_DIR}/evaluations.npz"
for _p in [_best_zip, _best_norm, _eval_npz]:
    _backup(_p)

print(f"\nWall time  : {_wall:.0f}s ({_wall/60:.1f} min)")
print(f"Final model: {FINAL_MODEL}.zip  ({_kb(FINAL_MODEL+'.zip'):.0f} KB)")
print(f"Best model : {_best_zip}  ({_kb(_best_zip):.0f} KB)")
assert os.path.isfile(FINAL_MODEL + ".zip"), "Final model not saved!"

_df_inv = checkpoint_inventory()
print(f"\nCheckpoints saved: {len(_df_inv)}")
print(_df_inv.to_string(index=False))
print("✅ Cell 16 — training complete")


In [ ]:
# Cell 17 — Resume training from final checkpoint only
# Loads saved VecNormalize from disk (no in-memory training state reused).
# Requires: DRY_RUN=False AND TRAIN_APPROVED=True

assert not DRY_RUN and TRAIN_APPROVED, (
    "Training blocked (DRY_RUN or TRAIN_APPROVED guard)."
)

_rmp, _rnp = _resolve_model_norm()

assert os.path.isfile(_rmp + ".zip"), f"Model not found: {_rmp}.zip"
assert os.path.isfile(_rnp),          f"Norm not found: {_rnp}"

print(f"Resuming from : {_rmp}.zip")
print(f"Norm          : {_rnp}")

# Build fresh train/eval pipelines from disk
_resume_train = DummyVecEnv([make_train_env(i) for i in range(N_ENVS)])
_resume_train = VecFrameStack(_resume_train, n_stack=N_STACK)
_resume_train = VecNormalize.load(_rnp, _resume_train)
_resume_train.training    = True
_resume_train.norm_reward = True

_resume_eval = build_eval_vec(seed=999)

from stable_baselines3 import PPO as _PPO2
_resume_model = _PPO2.load(_rmp, env=_resume_train)

_remaining = TOTAL_TIMESTEPS - _resume_model.num_timesteps
print(f"Completed     : {_resume_model.num_timesteps:,}")
print(f"Remaining     : {_remaining:,}")

if _remaining <= 0:
    print("No remaining steps. Skipping training.")
else:
    _rcb = make_callbacks(_resume_train, _resume_eval)
    try:
        _resume_model.learn(
            total_timesteps     = _remaining,
            reset_num_timesteps = False,
            progress_bar        = True,
            callback            = _rcb,
        )
    except KeyboardInterrupt:
        print("\n[INTERRUPTED] Saving...")

    _RMODEL = f"{CKPT_DIR}/model_resumed"
    _RNORM  = f"{CKPT_DIR}/vecnorm_resumed.pkl"
    _resume_model.save(_RMODEL)
    _resume_train.save(_RNORM)
    _backup(_RMODEL + ".zip")
    _backup(_RNORM)
    print(f"Saved: {_RMODEL}.zip")
    print("✅ Cell 17 — resume complete")


In [ ]:
# Cell 18 — Reload-after-disconnect smoke test
# After runtime restart: run Cell 1 (restart) -> Cells 2-10, then this cell.
# Loads best_model + best_vecnormalize from disk, verifies obs shape and action.

from stable_baselines3 import PPO as _PPO3

_rmp, _rnp = _resolve_model_norm()

assert os.path.isfile(_rmp + ".zip"), (
    f"Model not found: {_rmp}.zip\n"
    "Run Cell 16 first, or check that CKPT_DIR/BEST_DIR are correct."
)
assert os.path.isfile(_rnp), f"Norm not found: {_rnp}"

_vec = build_eval_vec_from_disk(_rnp, seed=42)
_m   = _PPO3.load(_rmp, env=_vec)

_obs = _vec.reset()
assert _obs.shape[-1] == 14 * N_STACK, (
    f"Obs dim mismatch: got {_obs.shape[-1]}, expected {14*N_STACK}. "
    "Wrong checkpoint? Older normalization files are incompatible with this final model."
)
_a, _ = _m.predict(_obs, deterministic=True)
assert _a.shape == (1, 3), f"Action shape {_a.shape}"
_vec.close()
del _vec, _m, _obs, _a

print(f"✅ Cell 18 — reload OK")
print(f"   Model : {os.path.basename(_rmp)}.zip")
print(f"   Obs   : ({14*N_STACK},)  Action: (3,)")


In [ ]:
# Cell 19 — Plot training/eval curves


def plot_training_curve(npz_path: str = None, save_path: str = None):
    """Mean +/- std eval reward from evaluations.npz."""
    npz_path = npz_path or f"{BEST_DIR}/evaluations.npz"
    if not os.path.isfile(npz_path):
        print(f"WARNING: evaluations.npz not found: {npz_path}")
        return
    d = np.load(npz_path)
    ts, res = d["timesteps"], d["results"]
    means, stds = res.mean(axis=1), res.std(axis=1)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ts, means, "b-o", ms=4, label="mean eval reward")
    ax.fill_between(ts, means - stds, means + stds, alpha=0.2)
    ax.set(xlabel="Timesteps", ylabel="Eval Reward (shaped)",
           title="CarRacing-v3 PPO Training Curve")
    ax.grid(True, ls="--", alpha=0.5)
    ax.legend()
    plt.tight_layout()
    p = save_path or f"{FIG_DIR}/training_curve.png"
    plt.savefig(p, dpi=120)
    _backup(p)
    plt.show()
    print(f"Saved: {p}")


def plot_multiseed_reward_bar(df: "pd.DataFrame", save_path: str = None):
    """Bar chart of raw reward per seed, colored by seed_type."""
    if df is None or df.empty:
        print("WARNING: No eval data for bar chart.")
        return
    colors = ["#2196F3" if str(t) == "compare5" else "#FF9800"
              for t in df.get("seed_type", [""] * len(df))]
    fig, ax = plt.subplots(figsize=(max(6, len(df)), 4))
    ax.bar(df["seed"].astype(str), df["raw_reward"], color=colors)
    mean_r = df["raw_reward"].mean()
    ax.axhline(mean_r, color="red", ls="--", label=f"mean={mean_r:.0f}")
    ax.set(xlabel="Seed", ylabel="Raw Reward", title="Multi-seed Evaluation")
    ax.legend()
    plt.tight_layout()
    p = save_path or f"{FIG_DIR}/multiseed_rewards.png"
    plt.savefig(p, dpi=120)
    _backup(p)
    plt.show()
    print(f"Saved: {p}")


def plot_failure_taxonomy(df: "pd.DataFrame", save_path: str = None):
    """Pie chart of done_reason distribution."""
    if df is None or df.empty or "done_reason" not in df.columns:
        print("WARNING: No done_reason data.")
        return
    counts = df["done_reason"].value_counts()
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(counts, labels=counts.index, autopct="%1.0f%%")
    ax.set_title("Episode termination reasons")
    plt.tight_layout()
    p = save_path or f"{FIG_DIR}/failure_taxonomy.png"
    plt.savefig(p, dpi=120)
    _backup(p)
    plt.show()


def plot_action_stats(df: "pd.DataFrame", save_path: str = None):
    """Grouped bar chart of mean action stats across seeds."""
    if df is None or df.empty:
        print("WARNING: No data for action stats chart.")
        return
    cols = [c for c in ["steer_abs_mean", "gas_mean", "brake_mean"]
            if c in df.columns]
    if not cols:
        return
    x = np.arange(len(df))
    w = 0.25
    fig, ax = plt.subplots(figsize=(max(6, len(df) + 2), 4))
    for i, col in enumerate(cols):
        ax.bar(x + i * w, df[col], w, label=col)
    ax.set_xticks(x + w)
    ax.set_xticklabels(df["seed"].astype(str))
    ax.set(xlabel="Seed", ylabel="Mean value", title="Action stats per seed")
    ax.legend()
    plt.tight_layout()
    p = save_path or f"{FIG_DIR}/action_stats.png"
    plt.savefig(p, dpi=120)
    _backup(p)
    plt.show()


# Auto-run training curve if evaluations.npz exists
plot_training_curve()
print("✅ Cell 19 — plot helpers defined")


In [ ]:
# Cell 20 — Final multi-seed evaluation

# Purpose:

# Run corrected independent multi-seed evaluation for final submission.

#

# This cell:

# - evaluates COMPARE5 fixed seeds

# - evaluates RANDOM10 deterministic random seeds

# - saves clean CSV files

# - saves one clean multi-seed reward bar chart

# - does NOT generate pie charts or debug-only action plots

#

# Requirements:

# - Cell 9 evaluation helpers must be corrected:

# model = _PPO.load(model_path)

# not:

# model = _PPO.load(model_path, env=vec)

# - Do NOT run training here.

import os
import random as _rnd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_mp, _np = artifact_sanity(prefer="best")

assert os.path.isfile(_mp + ".zip"), f"Model not found: {_mp}.zip"
assert os.path.isfile(_np), f"Norm not found: {_np}"

print("\n============================================================")
print("FINAL MULTI-SEED EVALUATION")
print("============================================================")
print(f"Model : {_mp}.zip")
print(f"Norm  : {_np}")

# ------------------------------

# COMPARE5

# ------------------------------

print("\n[1] COMPARE5 fixed-seed evaluation")
print("------------------------------------------------------------")
print(f"Seeds: {COMPARE5_SEEDS}")

seed_diversity_diagnostic(COMPARE5_SEEDS, norm_path=_np, strict=True)

compare5_rows = [
_run_eval_episode(
model_path=_mp,
norm_path=_np,
seed=int(_seed),
seed_type="compare5",
max_steps=1500,
)
for _seed in COMPARE5_SEEDS
]

_assert_unique_hashes(compare5_rows, "COMPARE5")

df_c5 = pd.DataFrame(compare5_rows)

print("\nCOMPARE5 results:")
display(df_c5[[
"seed",
"raw_reward",
"shaped_reward",
"episode_frames",
"done_reason",
"first_frame_hash",
]])

print(f"COMPARE5 mean raw : {df_c5['raw_reward'].mean():.1f} +/- {df_c5['raw_reward'].std():.1f}")
print(f"COMPARE5 min/max  : {df_c5['raw_reward'].min():.1f} / {df_c5['raw_reward'].max():.1f}")

_c5_csv = f"{TABLE_DIR}/compare5_results.csv"
df_c5.to_csv(_c5_csv, index=False)
_backup(_c5_csv)
print(f"Saved: {_c5_csv}")

# ------------------------------

# RANDOM10

# ------------------------------

print("\n[2] RANDOM10 deterministic random-seed evaluation")
print("------------------------------------------------------------")

_rng_r10 = _rnd.Random(RANDOM10_MASTER_SEED)
_r10_seeds = [_rng_r10.randint(0, 2**31 - 1) for _ in range(RANDOM10_COUNT)]

print(f"Master seed: {RANDOM10_MASTER_SEED}")
print(f"Seeds      : {_r10_seeds}")

seed_diversity_diagnostic(_r10_seeds, norm_path=_np, strict=True)

r10_rows = [
_run_eval_episode(
model_path=_mp,
norm_path=_np,
seed=int(_seed),
seed_type="random10",
max_steps=1500,
)
for _seed in _r10_seeds
]

_assert_unique_hashes(r10_rows, "RANDOM10")

df_r10 = pd.DataFrame(r10_rows)

print("\nRANDOM10 results:")
display(df_r10[[
"seed",
"raw_reward",
"shaped_reward",
"episode_frames",
"done_reason",
"first_frame_hash",
]])

print(f"RANDOM10 mean raw : {df_r10['raw_reward'].mean():.1f} +/- {df_r10['raw_reward'].std():.1f}")
print(f"RANDOM10 min/max  : {df_r10['raw_reward'].min():.1f} / {df_r10['raw_reward'].max():.1f}")

_r10_csv = f"{TABLE_DIR}/random10_results.csv"
df_r10.to_csv(_r10_csv, index=False)
_backup(_r10_csv)
print(f"Saved: {_r10_csv}")

# ------------------------------

# Clean combined summary table

# ------------------------------

print("\n[3] Combined final evaluation summary")
print("------------------------------------------------------------")

df_eval_all = pd.concat([df_c5, df_r10], ignore_index=True)

display(df_eval_all[[
"seed_type",
"seed",
"raw_reward",
"shaped_reward",
"episode_frames",
"done_reason",
"first_frame_hash",
]])

_summary_rows = [
{
"split": "COMPARE5",
"n": len(df_c5),
"mean_raw": round(float(df_c5["raw_reward"].mean()), 1),
"std_raw": round(float(df_c5["raw_reward"].std()), 1),
"min_raw": round(float(df_c5["raw_reward"].min()), 1),
"max_raw": round(float(df_c5["raw_reward"].max()), 1),
},
{
"split": "RANDOM10",
"n": len(df_r10),
"mean_raw": round(float(df_r10["raw_reward"].mean()), 1),
"std_raw": round(float(df_r10["raw_reward"].std()), 1),
"min_raw": round(float(df_r10["raw_reward"].min()), 1),
"max_raw": round(float(df_r10["raw_reward"].max()), 1),
},
]

df_eval_summary = pd.DataFrame(_summary_rows)

display(df_eval_summary)

_eval_all_csv = f"{TABLE_DIR}/final_multiseed_all_results.csv"
_eval_summary_csv = f"{TABLE_DIR}/final_multiseed_summary.csv"

df_eval_all.to_csv(_eval_all_csv, index=False)
df_eval_summary.to_csv(_eval_summary_csv, index=False)

_backup(_eval_all_csv)
_backup(_eval_summary_csv)

print(f"Saved: {_eval_all_csv}")
print(f"Saved: {_eval_summary_csv}")

# ------------------------------

# Clean final multi-seed reward chart

# ------------------------------

print("\n[4] Clean multi-seed reward chart")
print("------------------------------------------------------------")

_fig_dir = globals().get("FIG_DIR", globals().get("FIGURE_DIR", "/content/carracingv3_ppo_final/figures"))
os.makedirs(_fig_dir, exist_ok=True)

df_plot = df_eval_all.copy()
df_plot["label"] = df_plot["seed_type"].astype(str) + "\n" + df_plot["seed"].astype(str)

plt.figure(figsize=(14, 5))
plt.bar(df_plot["label"], df_plot["raw_reward"])
plt.axhline(900, linestyle="--", linewidth=1, label="High-score reference: 900")
plt.axhline(df_r10["raw_reward"].mean(), linestyle=":", linewidth=1, label="RANDOM10 mean")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Raw reward")
plt.title("Final Multi-seed Evaluation: PPO CarRacing-v3")
plt.legend()
plt.tight_layout()

_multiseed_clean_png = f"{_fig_dir}/final_multiseed_rewards_clean.png"
plt.savefig(_multiseed_clean_png, dpi=160)
plt.show()

_backup(_multiseed_clean_png)
print(f"Saved: {_multiseed_clean_png}")

print("\n============================================================")
print("FINAL MULTI-SEED EVALUATION COMPLETE")
print("Key result:")
print(f"  COMPARE5 : {df_c5['raw_reward'].mean():.1f} +/- {df_c5['raw_reward'].std():.1f}")
print(f"  RANDOM10 : {df_r10['raw_reward'].mean():.1f} +/- {df_r10['raw_reward'].std():.1f}")
print("============================================================")


In [ ]:
# Cell 21 — Final presentation video export and direct embedded preview

import os
import json
import base64
import numpy as np
import pandas as pd
from IPython.display import HTML, display

_mp, _np = artifact_sanity(prefer="best")

assert os.path.isfile(_mp + ".zip"), f"Model not found: {_mp}.zip"
assert os.path.isfile(_np), f"Norm not found: {_np}"
assert "df_c5" in globals(), "df_c5 not found. Run final Cell 20 first."
assert "df_r10" in globals(), "df_r10 not found. Run final Cell 20 first."

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

_tmp_c5 = df_c5.copy()
_tmp_c5["source_table"] = "compare5"

_tmp_r10 = df_r10.copy()
_tmp_r10["source_table"] = "random10"

df_video_candidates = pd.concat([_tmp_c5, _tmp_r10], ignore_index=True)
df_video_candidates = df_video_candidates.drop_duplicates(subset=["seed"], keep="first")

SUCCESS_FPS = 10
FAILURE_FPS = 10
FAILURE_REPEAT = 2
PAUSE_SECONDS = 2

print("============================================================")
print("FINAL VIDEO EXPORT — DIRECT DISPLAY VERSION")
print("============================================================")

# --- Select 3 success cases ---
success_done = df_video_candidates[
    (df_video_candidates["raw_reward"] >= 850) &
    (df_video_candidates["done_reason"].astype(str) == "env_done")
].copy()

success_done = success_done.sort_values(["episode_frames", "raw_reward"], ascending=[False, False])

success_high = df_video_candidates[
    (df_video_candidates["raw_reward"] >= 850) &
    (~df_video_candidates["seed"].isin(success_done["seed"]))
].copy()

success_selected = pd.concat([success_done, success_high], ignore_index=True).head(3)

if len(success_selected) < 3:
    fill = df_video_candidates[
        ~df_video_candidates["seed"].isin(success_selected["seed"])
    ].sort_values(["raw_reward", "episode_frames"], ascending=[False, False])
    success_selected = pd.concat([success_selected, fill], ignore_index=True).head(3)

success_selected = success_selected.reset_index(drop=True)

# --- Select 3 failure cases ---
failure_selected = df_video_candidates.sort_values(["raw_reward", "episode_frames"], ascending=[True, True]).head(3).reset_index(drop=True)

print("\n[1] Selected success cases")
display(success_selected[["source_table", "seed", "raw_reward", "episode_frames", "done_reason", "first_frame_hash"]])

print("\n[2] Selected failure cases")
display(failure_selected[["source_table", "seed", "raw_reward", "episode_frames", "done_reason", "first_frame_hash"]])

def _display_mp4_embedded(path, width=720):
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("ascii")
    display(HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{encoded}" type="video/mp4"></video>'))

def _save_meta(meta, path):
    with open(path, "w") as f: json.dump(meta, f, indent=2)

def _black_pause(frame, fps, seconds):
    return [np.zeros_like(frame)] * int(fps * seconds)

def _save_video_checked(frames, path, fps):
    save_video(frames, path, fps=fps)
    return path

print("\n[3] Recording success videos")
success_reel_frames = []
success_meta_list = []
for i, row in success_selected.iterrows():
    clip_id = i + 1
    seed = int(row["seed"])
    print(f"Recording SUCCESS {clip_id}/3 | seed={seed}")
    frames, meta = record_episode_fixed(model_path=_mp, norm_path=_np, seed=seed, max_steps=1500)
    duration = len(frames) / SUCCESS_FPS
    video_path = f"{VIDEO_DIR}/final_success_case_{clip_id:02d}_seed{seed}.mp4"
    _save_video_checked(frames, video_path, fps=SUCCESS_FPS)
    meta_out = {**meta, "video_type": "success_case", "approx_seconds": round(duration, 2), "video_path": video_path}
    _save_meta(meta_out, video_path.replace(".mp4", "_metadata.json"))
    if len(success_reel_frames) > 0: success_reel_frames.extend(_black_pause(frames[0], SUCCESS_FPS, PAUSE_SECONDS))
    success_reel_frames.extend(frames)
    success_meta_list.append(meta_out)

success_reel_path = f"{VIDEO_DIR}/final_success_reel.mp4"
_save_video_checked(success_reel_frames, success_reel_path, fps=SUCCESS_FPS)

print("\n[4] Recording failure videos")
failure_reel_frames = []
failure_meta_list = []
for i, row in failure_selected.iterrows():
    clip_id = i + 1
    seed = int(row["seed"])
    print(f"Recording FAILURE {clip_id}/3 | seed={seed}")
    frames, meta = record_episode_fixed(model_path=_mp, norm_path=_np, seed=seed, max_steps=1500)
    frames_out = frames * FAILURE_REPEAT
    duration = len(frames_out) / FAILURE_FPS
    video_path = f"{VIDEO_DIR}/final_failure_case_{clip_id:02d}_seed{seed}.mp4"
    _save_video_checked(frames_out, video_path, fps=FAILURE_FPS)
    meta_out = {**meta, "video_type": "failure_case", "approx_seconds": round(duration, 2), "video_path": video_path}
    _save_meta(meta_out, video_path.replace(".mp4", "_metadata.json"))
    if len(failure_reel_frames) > 0: failure_reel_frames.extend(_black_pause(frames_out[0], FAILURE_FPS, PAUSE_SECONDS))
    failure_reel_frames.extend(frames_out)
    failure_meta_list.append(meta_out)

failure_reel_path = f"{VIDEO_DIR}/final_failure_reel.mp4"
_save_video_checked(failure_reel_frames, failure_reel_path, fps=FAILURE_FPS)

print("\n[6] Previews")
_display_mp4_embedded(success_reel_path)
_display_mp4_embedded(failure_reel_path)

In [ ]:
# Cell 22 — Final deliverable review

import os
import glob
import base64
import pandas as pd
from IPython.display import display, Image as IPyImage, HTML

print("============================================================")
print("FINAL DELIVERABLE REVIEW")
print("============================================================")

# --- Resolve directories ---
_fig_dir = globals().get("FIG_DIR", "/content/carracingv3_ppo_final/figures")
_video_dir = globals().get("VIDEO_DIR", "/content/carracingv3_ppo_final/videos")
_table_dir = globals().get("TABLE_DIR", "/content/carracingv3_ppo_final/tables")
_best_dir = globals().get("BEST_DIR", "/content/carracingv3_ppo_final/models/best")
_ckpt_dir = globals().get("CKPT_DIR", "/content/carracingv3_ppo_final/models/checkpoints")

# --- Helpers ---
def _latest_file(pattern):
    files = glob.glob(pattern)
    if len(files) == 0: return None
    return max(files, key=os.path.getmtime)

def _print_file_status(label, path):
    if path is not None and os.path.isfile(path):
        print(f"OK   {label}: {path}  ({os.path.getsize(path) / 1024:.1f} KB)")
        return True
    print(f"MISS {label}: {path}")
    return False

def _display_mp4_embedded(path, width=720):
    if path is None or not os.path.isfile(path):
        print(f"MISS video preview: {path}")
        return
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("ascii")
    display(HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{encoded}" type="video/mp4"></video>'))

# 1. Final numerical results
print("\n[1] Final numerical results")
print("------------------------------------------------------------")
_summary_csv = f"{_table_dir}/final_multiseed_summary.csv"
_all_csv = f"{_table_dir}/final_multiseed_all_results.csv"

_print_file_status("summary", _summary_csv)
if os.path.isfile(_summary_csv): display(pd.read_csv(_summary_csv))

_print_file_status("all_results", _all_csv)
if os.path.isfile(_all_csv): display(pd.read_csv(_all_csv).head(5))

# 2. Core figures
print("\n[2] Core figures")
print("------------------------------------------------------------")
_training_curve = f"{_fig_dir}/training_curve.png"
_clean_multiseed = f"{_fig_dir}/final_multiseed_rewards_clean.png"

if _print_file_status("training_curve", _training_curve): display(IPyImage(filename=_training_curve))
if _print_file_status("multiseed_rewards", _clean_multiseed): display(IPyImage(filename=_clean_multiseed))

# 3. Final videos
print("\n[3] Final presentation videos")
print("------------------------------------------------------------")
_success_reel = f"{_video_dir}/final_success_reel.mp4"
_failure_reel = f"{_video_dir}/final_failure_reel.mp4"

_print_file_status("SUCCESS REEL", _success_reel)
_print_file_status("FAILURE REEL", _failure_reel)

print("\nSUCCESS REEL PREVIEW")
_display_mp4_embedded(_success_reel)

print("\nFAILURE REEL PREVIEW")
_display_mp4_embedded(_failure_reel)

# 4. Artifacts & Readiness
print("\n[4] Readiness check")
_required = [_summary_csv, _all_csv, _training_curve, _success_reel, f"{_best_dir}/best_model.zip"]
_missing = [p for p in _required if not os.path.isfile(p)]

if not _missing:
    print("READY: All deliverables present. Proceed to Cell 23.")
else:
    print(f"NOT READY: Missing {_missing}")

print("\n============================================================")
print("FINAL REVIEW COMPLETE")
print("============================================================")

In [ ]:
# Cell 23 — Clean final release package

# Purpose:
# Create one clean final submission package with a simple filename.

import os
import glob
import json
import shutil
import zipfile
import pandas as pd
from pathlib import Path

print("============================================================")
print("CLEAN FINAL RELEASE PACKAGE")
print("============================================================")

# ------------------------------------------------------------
# Resolve directories
# ------------------------------------------------------------
ROOT_DIR = globals().get("ROOT_DIR", "/content/carracingv3_ppo_final")
TABLE_DIR = globals().get("TABLE_DIR", f"{ROOT_DIR}/tables")
FIG_DIR = globals().get("FIG_DIR", globals().get("FIGURE_DIR", f"{ROOT_DIR}/figures"))
VIDEO_DIR = globals().get("VIDEO_DIR", f"{ROOT_DIR}/videos")
BEST_DIR = globals().get("BEST_DIR", f"{ROOT_DIR}/models/best")

RELEASE_DIR = "/content/CarRacingV3_PPO_Final_Submission"
ZIP_BASE = "/content/CarRacingV3_PPO_Final_Submission"
ZIP_PATH = ZIP_BASE + ".zip"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/CarRacingV3_PPO_Final_Submission.zip"

# Reset release directory
if os.path.isdir(RELEASE_DIR):
    shutil.rmtree(RELEASE_DIR)

os.makedirs(RELEASE_DIR, exist_ok=True)
os.makedirs(f"{RELEASE_DIR}/tables", exist_ok=True)
os.makedirs(f"{RELEASE_DIR}/figures", exist_ok=True)
os.makedirs(f"{RELEASE_DIR}/videos", exist_ok=True)
os.makedirs(f"{RELEASE_DIR}/models", exist_ok=True)
os.makedirs(f"{RELEASE_DIR}/metadata", exist_ok=True)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
copied_files = []

def latest_file(pattern):
    files = glob.glob(pattern)
    if len(files) == 0:
        return None
    return max(files, key=os.path.getmtime)

def copy_required(src, dst_dir, label):
    assert src is not None and os.path.isfile(src), f"Missing required file: {label} -> {src}"
    dst = os.path.join(dst_dir, os.path.basename(src))
    shutil.copy2(src, dst)
    copied_files.append(dst)
    print(f"OK   {label}: {dst}")
    return dst

def copy_optional(src, dst_dir, label):
    if src is None or not os.path.isfile(src):
        print(f"SKIP {label}: {src}")
        return None
    dst = os.path.join(dst_dir, os.path.basename(src))
    shutil.copy2(src, dst)
    copied_files.append(dst)
    print(f"OK   {label}: {dst}")
    return dst

def copy_glob(pattern, dst_dir, label):
    files = sorted(glob.glob(pattern))
    if len(files) == 0:
        print(f"SKIP {label}: no files matched {pattern}")
        return []
    out = []
    for src in files:
        if os.path.isfile(src):
            dst = os.path.join(dst_dir, os.path.basename(src))
            shutil.copy2(src, dst)
            copied_files.append(dst)
            out.append(dst)
    print(f"OK   {label}: copied {len(out)} files")
    return out

def record_file(path, category):
    return {
        "category": category,
        "filename": os.path.basename(path),
        "relative_path": os.path.relpath(path, RELEASE_DIR),
        "size_kb": round(os.path.getsize(path) / 1024, 1),
    }

# ------------------------------------------------------------
# Resolve required artifacts
# ------------------------------------------------------------
summary_csv = f"{TABLE_DIR}/final_multiseed_summary.csv"
all_results_csv = f"{TABLE_DIR}/final_multiseed_all_results.csv"
compare5_csv = f"{TABLE_DIR}/compare5_results.csv"
random10_csv = f"{TABLE_DIR}/random10_results.csv"
video_index_csv = f"{TABLE_DIR}/final_video_index.csv"
training_curve = f"{FIG_DIR}/training_curve.png"
multiseed_clean = f"{FIG_DIR}/final_multiseed_rewards_clean.png"
success_reel = latest_file(f"{VIDEO_DIR}/final_success_reel*.mp4")
failure_reel = latest_file(f"{VIDEO_DIR}/final_failure_reel*.mp4")
best_model = f"{BEST_DIR}/best_model.zip"
best_norm = f"{BEST_DIR}/best_vecnormalize.pkl"
eval_npz = f"{BEST_DIR}/evaluations.npz"

# ------------------------------------------------------------
# Copy tables
# ------------------------------------------------------------
print("\n[1] Copy final tables")
copy_required(summary_csv, f"{RELEASE_DIR}/tables", "final summary")
copy_required(all_results_csv, f"{RELEASE_DIR}/tables", "all multi-seed results")
copy_required(compare5_csv, f"{RELEASE_DIR}/tables", "COMPARE5 results")
copy_required(random10_csv, f"{RELEASE_DIR}/tables", "RANDOM10 results")
copy_optional(video_index_csv, f"{RELEASE_DIR}/tables", "video index")

# ------------------------------------------------------------
# Copy figures
# ------------------------------------------------------------
print("\n[2] Copy final figures")
copy_required(training_curve, f"{RELEASE_DIR}/figures", "training curve")
copy_required(multiseed_clean, f"{RELEASE_DIR}/figures", "clean multi-seed reward chart")

# ------------------------------------------------------------
# Copy videos
# ------------------------------------------------------------
print("\n[3] Copy final videos")
copy_required(success_reel, f"{RELEASE_DIR}/videos", "success reel")
copy_required(failure_reel, f"{RELEASE_DIR}/videos", "failure reel")
copy_glob(f"{VIDEO_DIR}/final_success_case_[0-9][0-9]**.mp4", f"{RELEASE_DIR}/videos", "individual success cases")
copy_glob(f"{VIDEO_DIR}/final_failure_case*[0-9][0-9]_*.mp4", f"{RELEASE_DIR}/videos", "individual failure cases")
copy_glob(f"{VIDEO_DIR}/final_success_case_[0-9][0-9]***metadata.json", f"{RELEASE_DIR}/metadata", "success case metadata")
copy_glob(f"{VIDEO_DIR}/final_failure_case*[0-9][0-9]**_metadata.json", f"{RELEASE_DIR}/metadata", "failure case metadata")
copy_glob(f"{VIDEO_DIR}/final_success_reel*_metadata.json", f"{RELEASE_DIR}/metadata", "success reel metadata")
copy_glob(f"{VIDEO_DIR}/final_failure_reel*_metadata.json", f"{RELEASE_DIR}/metadata", "failure reel metadata")

# ------------------------------------------------------------
# Copy model artifacts
# ------------------------------------------------------------
print("\n[4] Copy model artifacts")
copy_required(best_model, f"{RELEASE_DIR}/models", "best PPO model")
copy_required(best_norm, f"{RELEASE_DIR}/models", "best VecNormalize")
copy_required(eval_npz, f"{RELEASE_DIR}/models", "EvalCallback evaluations")

# ------------------------------------------------------------
# Write README
# ------------------------------------------------------------
print("\n[5] Write README")
readme_path = f"{RELEASE_DIR}/README.md"
readme_text = """# CarRacing-v3 PPO Final Submission\n\n## Project Summary\nThis project trained a PPO-based CarRacing-v3 agent using compact visual-feature extraction.\n"""
with open(readme_path, "w") as f:
    f.write(readme_text)
copied_files.append(readme_path)

# ------------------------------------------------------------
# Write manifest
# ------------------------------------------------------------
print("\n[6] Write manifest")
manifest_rows = []
for path in copied_files:
    if path is not None and os.path.isfile(path):
        category = Path(path).parent.name
        manifest_rows.append(record_file(path, category))
manifest_df = pd.DataFrame(manifest_rows)
manifest_path = f"{RELEASE_DIR}/MANIFEST.csv"
manifest_df.to_csv(manifest_path, index=False)
display(manifest_df)

# ------------------------------------------------------------
# Create zip
# ------------------------------------------------------------
print("\n[7] Create clean zip")
if os.path.isfile(ZIP_PATH):
    os.remove(ZIP_PATH)
shutil.make_archive(ZIP_BASE, "zip", RELEASE_DIR)
print(f"OK   Clean zip created: {ZIP_PATH}")

# ------------------------------------------------------------
# Optional copy to Google Drive
# ------------------------------------------------------------
print("\n[9] Optional copy to Google Drive")
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy2(ZIP_PATH, DRIVE_ZIP_PATH)
    print(f"OK   Copied to Drive: {DRIVE_ZIP_PATH}")
else:
    print("Drive is not mounted.")

print("\n============================================================")
print("CLEAN FINAL SUBMISSION READY")
print("============================================================")